In [ ]:
import torch, time
from transformers import AutoTokenizer, AutoModelForCausalLM

# --- 1. Set up model and tokenizer ---
# Define the model ID from Hugging Face Hub
model_id = "google/gemma-3-1b-it"

# Set the device to use (GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Define the data type for model weights (bfloat16 for better performance on modern GPUs)
torch_dtype = torch.bfloat16

# Load the tokenizer
# The tokenizer converts your text prompt into a format the model understands (tokens)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load the model
# device_map="auto" lets accelerate handle memory distribution
# torch_dtype sets the precision of the model's weights to save memory and speed up inference
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch_dtype,
    device_map="cuda",
)

# --- 2. Create the prompt using the chat template ---
# Instruction-tuned models work best when the input follows a specific format.
# The tokenizer's `apply_chat_template` method handles this for you.
chat = [
    { "role": "user", 
     "content": """Write a 200 words essay"""},
]

# Apply the template, convert to PyTorch tensors, and move to the designated device
prompt_tokens = tokenizer.apply_chat_template(chat, tokenize=True, add_generation_prompt=True, return_tensors="pt")
input_ids = prompt_tokens.to(device)


# --- Generate and Benchmark the response ---
print("\nGenerating response...")

# <-- 2. Start the timer right before generation
start_time = time.perf_counter()

outputs = model.generate(
    input_ids,
    max_new_tokens=150,
)

# <-- 3. Stop the timer right after generation
end_time = time.perf_counter()


# --- Decode the result (same as before) ---
response_text = tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)


# --- 4. Calculate and print the performance ---
duration = end_time - start_time
num_input_tokens = input_ids.shape[1]
num_new_tokens = outputs.shape[1] - num_input_tokens
tokens_per_second = num_new_tokens / duration

print("-" * 20)
print(response_text)
print("-" * 20)
print(f"Generated {num_new_tokens} new tokens in {duration:.2f} seconds.")
print(f"Performance: {tokens_per_second:.2f} tokens/second  ")

Using device: cuda

Generating response...
--------------------
Okay, here's a 200-word essay on the importance of embracing change:

---

In a world defined by constant evolution, the ability to embrace change is no longer a desirable trait – it’s a fundamental necessity. We often cling to the familiar, fearing disruption and the unknown, but resisting progress ultimately stifles growth and limits our potential.  Change, whether it’s personal, societal, or technological, is the engine of innovation and adaptation. 

Historically, civilizations that thrived were those that were willing to challenge existing norms, to learn from new experiences, and to pivot in response to shifting circumstances.  Failure to adapt, conversely, often leads to stagnation and decline.  Embracing change doesn’
--------------------
